# Speech Denoising — Notebook 04: Training

Goal: train the `UNet` (from `model.py`) on the VoiceBank+DEMAND spectrograms,
using MSE loss and the Adam optimizer, with MPS (Apple Silicon GPU) acceleration.
The best checkpoint (by train loss) is saved to `models/best_model.pth`, then
verified against the held-out test set.

In [ ]:
from model import UNet
import torch
from torch import nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

## Load Preprocessed Spectrograms

In [ ]:
clean_testset_amp = np.load("data/processed/clean_testset_amp.npy")
clean_trainset_amp = np.load("data/processed/clean_trainset_amp.npy")
noisy_testset_amp = np.load("data/processed/noisy_testset_amp.npy")
noisy_trainset_amp = np.load("data/processed/noisy_trainset_amp.npy")

## Convert to PyTorch Tensors

In [ ]:
torch_clean_testset_amp = torch.from_numpy(clean_testset_amp).float()
torch_clean_trainset_amp = torch.from_numpy(clean_trainset_amp).float()
torch_noisy_testset_amp = torch.from_numpy(noisy_testset_amp).float()
torch_noisy_trainset_amp = torch.from_numpy(noisy_trainset_amp).float()

## Dataset & DataLoader

One training pair = (noisy spectrogram, clean spectrogram) at the same index.
`shuffle=True` only for training, so the model doesn't memorise file order.

In [ ]:
tensor_test_dataset = TensorDataset(torch_noisy_testset_amp, torch_clean_testset_amp)
tensor_train_dataset = TensorDataset(torch_noisy_trainset_amp, torch_clean_trainset_amp)
train_loader = DataLoader(tensor_train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(tensor_test_dataset, batch_size=32)

## Model, Loss, Optimizer

- **Loss**: MSE (Mean Squared Error) - averages the squared difference between
  predicted and target amplitude across every point in the spectrogram, so the
  result doesn't depend on batch size or spectrogram size.
- **Optimizer**: Adam - takes the gradients computed by backprop and updates
  the model's weights; adapts the step size per-weight based on gradient history.
- **Device**: `mps` - Apple Silicon GPU acceleration (Metal Performance Shaders).

In [ ]:
model = UNet()
device = torch.device("mps")
model = model.to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Training Loop

For every batch: zero the gradients, forward pass, compute loss, backward pass,
optimizer step. `noisy_batch`/`clean_batch` come out of the `DataLoader` without
a channel dimension - `.unsqueeze(1)` adds it back to match what `UNet` expects
(`batch, channels, height, width`).

The checkpoint with the lowest train loss so far is saved to
`models/best_model.pth` after every epoch - a simple form of early-stopping
protection, in case loss started climbing back up on a later epoch.

A separate experiment (continuing training for 10 more epochs on top of this
checkpoint, 20 total) is documented in notebook 05 - train loss kept falling
but test loss and PESQ got worse, so this 10-epoch run was kept as final.

In [ ]:
num_epochs = 10
best_loss = float('inf')

for epoch in range(num_epochs):
    epoch_loss = 0
    for noisy_batch, clean_batch in tqdm(train_loader):
        optimizer.zero_grad()
        prediction = model(noisy_batch.unsqueeze(1).to(device))
        loss = loss_fn(prediction, clean_batch.unsqueeze(1).to(device))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    loss_fin = epoch_loss / len(train_loader)
    if loss_fin < best_loss:
        best_loss = loss_fin
        torch.save(model.state_dict(), "models/best_model.pth")

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss_fin}")

100%|██████████| 1239/1239 [07:22<00:00,  2.80it/s]


Epoch 1/10, Loss: 0.21279836569922503


100%|██████████| 1239/1239 [07:21<00:00,  2.81it/s]


Epoch 2/10, Loss: 0.17777437533614998


100%|██████████| 1239/1239 [07:21<00:00,  2.80it/s]


...

Epoch 8/10, Loss: 0.12270886052318693


100%|██████████| 1239/1239 [07:31<00:00,  2.75it/s]


Epoch 9/10, Loss: 0.11600840926362778


100%|██████████| 1239/1239 [07:27<00:00,  2.77it/s]


Epoch 10/10, Loss: 0.11399522912632178


## Test Set Evaluation

Loads the best saved checkpoint explicitly (so this cell gives the same result
regardless of what happens to stay in memory from training), then runs the
model on data it never saw during training.

`model.eval()` switches off training-only behaviour (e.g. BatchNorm uses its
running statistics instead of per-batch stats). `torch.no_grad()` skips
gradient tracking entirely, since no backward pass happens here.

In [ ]:
model.load_state_dict(torch.load("models/best_model.pth"))
model.eval()

test_loss = 0
with torch.no_grad():
    for noisy_test_batch, clean_test_batch in test_loader:
        prediction = model(noisy_test_batch.unsqueeze(1).to(device))
        loss = loss_fn(prediction, clean_test_batch.unsqueeze(1).to(device))
        test_loss += loss.item()

print(f"Test Loss: {test_loss/len(test_loader)}")

Test Loss: 0.044904967865500696
